<a href="https://colab.research.google.com/github/trash-taste/krisha-kz-mlops/blob/main/almaty_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏙️ Almaty Real Estate AVM (Automated Valuation Model)

![Python](https://img.shields.io/badge/python-3.9+-blue.svg)
![CatBoost](https://img.shields.io/badge/CatBoost-1.2-yellow.svg)
![Scikit-Learn](https://img.shields.io/badge/scikit--learn-1.3+-orange.svg)
![Status](https://img.shields.io/badge/Status-Production_Ready-success.svg)

## 📌 Business Problem
Оценка недвижимости в развивающихся рынках (Алматы) осложнена высоким уровнем шума в данных: фейковые объявления, экстремальные выбросы и отсутствие точных гео-координат.
**Цель проекта:** Создать интерпретируемую ML-систему (AVM) для автоматической оценки рыночной стоимости квартир, которую можно интегрировать в backend (API) для использования риелторами и клиентами.

## ⚙️ Architecture & Pipeline Design
Проект спроектирован по стандартам **Production ML**, строго разделяя трансформации без состояния и трансформации, зависящие от данных, для предотвращения Data Leakage (утечки данных).

1. **Stateless Preprocessing:** Санитарная фильтрация (удаление опечаток парсинга через IQR), приведение типов, создание базовых эвристик (proxy features: `near_metro`, `location_abay`).
2. **Strict Data Split:** Разделение данных ДО применения любых stateful-трансформаций.
3. **Stateful Feature Engineering (Custom Transformers):**
   - `SmoothedTargetEncoder`: Байесовское сглаживание таргета по микро-локациям для борьбы с переобучением на редких районах.
   - `PriceTierBinner`: Дискретизация закодированных локаций на 5 ценовых сегментов (Market Tiers).
4. **Outlier Isolation (Train Only):** Использование масштабированного `IsolationForest` для удаления многомерных аномалий исключительно из обучающей выборки (чтобы модель умела работать с "грязными" запросами в проде).
5. **Modeling:** Трансформация таргета (`price_per_sqm` вместо `total_price`) для снижения дисперсии. Основная модель — `CatBoostRegressor`.

## 📊 Model Benchmarking (5-Fold CV)
Перед выбором финальной модели был проведен бейзлайн-тест на очищенных данных:

| Model | CV MAPE (Score) | Std Dev |
| :--- | :--- | :--- |
| Linear Regression | 16.30% | ± 0.56% |
| Random Forest | 14.24% | ± 0.40% |
| **CatBoost (Champion)** | **13.70%** | **Holdout Test** |

*Бизнес-результат:* Модель ошибается в среднем на **13.7%** (MAE: ~4.3 млн ₸), что укладывается в стандарты "вилки для торга" живого риелтора (10-15%).

## 🧠 Explainability (SHAP Insights)
Модель не является "черным ящиком". Ключевые инсайты рынка Алматы (см. `notebooks/02_model_experiments.ipynb`):
* **Площадь:** Главный драйвер цены, но имеет нелинейную зависимость (скидка за огромные площади).
* **Location Premium:** Фича `location_abay` (Выше Абая) статистически доказала наличие мощной ценовой премии за экологию и престиж, обходя формальное деление на административные районы.
* **Метро:** Влияние `near_metro` оказалось минимальным, что подтверждает автомобилецентричность рынка Алматы.

## 📦 Deployment & Inference
Модель и все стейтфул-трансформеры упакованы в единый артефакт `avm_artifacts.pkl` через `joblib`.
Пример использования (см. `src/inference.py`):
```python
from src.inference import RealEstateAVM

avm = RealEstateAVM(artifact_path='models/avm_artifacts.pkl')
prediction = avm.predict({
    "plosh_ob": 55.0,
    "gorod": "Алматы",
    "rayon": "Бостандыкский",
    "ulica": "проспект Абая"
})
print(f"Оценка: {prediction['total_price']} KZT")

In [38]:
import os

# =========================
# 1. CHECK DATA
# =========================
if not os.path.exists("Data_2024_Almaty_kv.csv"):
    raise FileNotFoundError("Upload Data_2024_Almaty_kv.csv to Colab root")

# =========================
# 2. STRUCTURE
# =========================
!rm -rf almaty-avm
!mkdir -p almaty-avm/src almaty-avm/models almaty-avm/data_sample

!mv Data_2024_Almaty_kv.csv almaty-avm/data_sample/

# =========================
# 3. REQUIREMENTS
# =========================
with open("almaty-avm/requirements.txt", "w") as f:
    f.write("""pandas
numpy
scikit-learn
catboost
joblib
""")

# =========================
# 4. PREPROCESS
# =========================
with open("almaty-avm/src/preprocess.py", "w", encoding="utf-8") as f:
    f.write("""
import pandas as pd
import numpy as np

class DataPreprocessor:
    def transform(self, df, is_train=False):
        df = df.copy()

        for c in ["summa", "plosh_ob"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        if is_train:
            df = df.dropna(subset=["summa","plosh_ob"])
            df = df[df["plosh_ob"] >= 15]
            df["price_per_sqm"] = df["summa"] / df["plosh_ob"]

            q1, q3 = df["price_per_sqm"].quantile([0.25,0.75])
            iqr = q3 - q1
            df = df[(df["price_per_sqm"] >= 300000) &
                    (df["price_per_sqm"] <= q3 + 1.5*iqr)]

        df["near_metro"] = df["ulica"].astype(str).str.lower().apply(
            lambda x: int(any(s in x for s in ["абая","назарбаев","фурманова","гагарина"]))
        )

        df["location_abay"] = df["rayon"].astype(str).apply(
            lambda x: int(any(s in x for s in ["бостандыкский","медеуский","наурызбайский"]))
        )

        df["micro_loc"] = df["gorod"].astype(str) + "|" + df["rayon"].astype(str)

        return df
""")

# =========================
# 5. TRANSFORMERS
# =========================
with open("almaty-avm/src/transformers.py", "w", encoding="utf-8") as f:
    f.write("""
from sklearn.base import BaseEstimator, TransformerMixin

class SmoothedTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, col="micro_loc", alpha=15):
        self.col = col
        self.alpha = alpha

    def fit(self, X, y):
        self.global_mean = y.mean()
        stats = y.groupby(X[self.col]).agg(["mean","count"])
        self.mapping = (stats["mean"] * stats["count"] + self.global_mean * self.alpha) / (stats["count"] + self.alpha)
        return self

    def transform(self, X):
        X = X.copy()
        X[self.col+"_enc"] = X[self.col].map(self.mapping).fillna(self.global_mean)
        return X
""")

# =========================
# 6. TRAIN
# =========================
with open("almaty-avm/src/train.py", "w", encoding="utf-8") as f:
    f.write("""
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor

from preprocess import DataPreprocessor
from transformers import SmoothedTargetEncoder

print("TRAINING START")

df = pd.read_csv("../data_sample/Data_2024_Almaty_kv.csv")

df = DataPreprocessor().transform(df, is_train=True)

X = df[["plosh_ob","gorod","rayon","micro_loc","near_metro","location_abay"]]
y = df["price_per_sqm"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = Pipeline([
    ("enc", SmoothedTargetEncoder()),
    ("model", CatBoostRegressor(iterations=800, depth=6, learning_rate=0.05, verbose=0))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)

mape = (abs(pred - y_test)/y_test).mean()*100
print("MAPE:", mape)

os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/model.pkl")
print("SAVED")
""")

# =========================
# 7. INFERENCE
# =========================
with open("almaty-avm/src/inference.py", "w", encoding="utf-8") as f:
    f.write("""
import joblib
import pandas as pd
from preprocess import DataPreprocessor

model = joblib.load("../models/model.pkl")
prep = DataPreprocessor()

def predict(data):
    df = prep.transform(pd.DataFrame([data]), is_train=False)
    pred = model.predict(df)[0]
    return {
        "price_per_sqm": int(pred),
        "total": int(pred * data["plosh_ob"])
    }

if __name__ == "__main__":
    print(predict({
        "plosh_ob": 60,
        "gorod": "Алматы",
        "rayon": "Бостандыкский",
        "ulica": "Абая"
    }))
""")

# =========================
# 8. RUN
# =========================
!pip install -q -r almaty-avm/requirements.txt

%cd almaty-avm/src
!python train.py
!python inference.py

%cd /content
!zip -r almaty-avm.zip almaty-avm

print("DONE -> almaty-avm.zip")

/content/almaty-avm/src
TRAINING START
Traceback (most recent call last):
  File "_catboost.pyx", line 2606, in _catboost.get_float_feature
  File "_catboost.pyx", line 1290, in _catboost._FloatOrNan
  File "_catboost.pyx", line 1039, in _catboost._FloatOrNanFromString
TypeError: Cannot convert ' р-н Ауэзовский' to float

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/almaty-avm/src/train.py", line 28, in <module>
    model.fit(X_train, y_train)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/usr/local/lib/python3.12/dist-packages/catboost/core.py", line 6178, in fit
    return self._fit(X, y, cat_features, text_featur